# 🚀 Notebook do Professor (Demo) — Aula 04: Context Engineering De prompt para contexto

**Disciplina:** Prompt Engineering and Artificial Intelligence  
**Instituição:** FIAP — Ciência da Computação · 2026  
**Professor:** Jorge Luiz Gomes  
**Aula 04/14 — Módulo 1: LangChain Foundations · 🏁 Entrega CKP01**  
**⏱️ 1h40min**  
**🧠 Context rot · XML tagging · Meta prompting**  
**🏁 CKP01 entrega**  

---

## 🎯 Objetivo da aula

Entender que o contexto é um recurso finito e caro — e aprender a preenchê-lo intencionalmente: certas informações no lugar certo, na hora certa, com o mínimo de tokens necessário. Aplicar isso ao CKP01.

---

## Como usar este notebook

- Cada célula corresponde a um slide de código da aula (a ordem é a da apresentação).
- Rode ao vivo enquanto explica o slide correspondente.
- A última seção traz o lab do aluno com o gabarito das lacunas.

---

# 🔬 Código da aula — slide a slide

In [ ]:
!pip install langchain-ollama langchain-core langchain-classic -q

from langchain_ollama import ChatOllama
from google.colab import userdata
import os

# Definir a API key via variável de ambiente (Colab Secrets)
os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

### Slide 10 — XML tagging — estruturar o contexto em seções

```
Você é um assistente de culinária
brasileira. Responda sempre em
português. Não fale sobre outros
assuntos. Se o usuário perguntar
sobre outro tema, redirecione. Seja
amigável e use emojis. Mencione
ingredientes locais. Limite respostas
a 3 parágrafos. Não use markdown.
Cite fontes quando possível...

# 8 instruções misturadas num bloco
# O modelo tende a priorizar as primeiras
# e "esquecer" as do meio e do fim
```

### Slide 10 — XML tagging — estruturar o contexto em seções

```
<persona>
  Assistente de culinária brasileira.
  Tom amigável com emojis moderados.
</persona>

<restricoes>
  - Somente tópicos de culinária
  - Redirecionar outros assuntos
  - Máximo 3 parágrafos por resposta
</restricoes>

<formato>
  Sem markdown. Sem headers.
  Ingredientes em listas simples.
</formato>

# Cada seção é processada como unidade
# Mais fácil de atualizar e depurar
```

### Slide 11 — Meta prompting — o modelo que melhora seus próprios prompts

In [ ]:
PROMPT_OTIMIZADOR = """
<tarefa>
Você é um especialista em context engineering.
Analise o system prompt abaixo e reescreva-o aplicando:
- XML tagging para separar seções
- Instruções críticas no início e no final
- Remoção de redundâncias
- Clareza nas restrições
</tarefa>

<prompt_original>
{prompt_original}
</prompt_original>

<formato_saida>
Retorne o prompt otimizado entre tags <prompt_otimizado>...</prompt_otimizado>
Depois explique em 3 bullet points as principais mudanças feitas.
</formato_saida>
"""

# Chain de meta prompting
chain_otimizador = ChatPromptTemplate.from_template(PROMPT_OTIMIZADOR) | llm | StrOutputParser()

# Usar: mandar o system prompt atual para ser melhorado
resultado = chain_otimizador.invoke({
    "prompt_original": MEU_SYSTEM_PROMPT_ATUAL
})
print(resultado)  # → versão melhorada + explicação das mudanças

### Slide 12 — Medir tokens — quanto o contexto está custando

In [ ]:
!pip install tiktoken -q

import tiktoken

def contar_tokens(texto: str, modelo: str = "gpt-4") -> int:
    """Conta tokens usando o tokenizador do modelo especificado."""
    enc = tiktoken.encoding_for_model(modelo)
    return len(enc.encode(texto))

# Comparar system prompt antes e depois de context engineering
ANTES = """Você é um assistente especializado em culinária brasileira.
Responda sempre em português do Brasil com um tom amigável e use emojis
moderadamente. Não fale sobre assuntos não relacionados a culinária.
Se o usuário perguntar sobre outro assunto, redirecione gentilmente.
Limite suas respostas a no máximo 3 parágrafos. Não use markdown.
Mencione ingredientes e técnicas tradicionais brasileiras sempre que
possível. Cite a região de origem dos pratos quando souber..."""

DEPOIS = """<persona>Chef assistente de culinária brasileira. Tom amigável, emojis moderados.</persona>
<restricoes>Somente culinária. Redirecionar outros temas. Máximo 3 parágrafos. Sem markdown.</restricoes>
<contexto>Priorizar ingredientes e técnicas regionais brasileiras.</contexto>"""

print(f"Antes:  {contar_tokens(ANTES)} tokens")   # → ~90 tokens
print(f"Depois: {contar_tokens(DEPOIS)} tokens")  # → ~50 tokens
print(f"Redução: {(1 - contar_tokens(DEPOIS)/contar_tokens(ANTES))*100:.0f}%")

### Slide 21 — Python novo desta aula

In [ ]:
# 1. tiktoken — contar tokens antes de enviar ao modelo
import tiktoken
enc = tiktoken.encoding_for_model("gpt-4")
tokens = enc.encode("Meu texto aqui")  # lista de IDs
len(tokens)                              # número de tokens

# 2. Strings multilinha com XML (melhor legibilidade)
SYSTEM = """<persona>
Chef de culinária. Tom amigável.
</persona>
<restricoes>
Somente culinária. Máximo 3 parágrafos.
</restricoes>"""

# 3. f-string com cálculo percentual inline
reducao = (1 - depois / antes) * 100
print(f"Redução: {reducao:.1f}%")  # :1f → 1 casa decimal

# 4. from_template() — para prompts sem roles
prompt = ChatPromptTemplate.from_template("""
Reescreva este prompt com XML tagging:
{prompt_original}
""")  # sem roles — tratado como HumanMessage

# 5. Função com type annotation de retorno (-> int)
def contar_tokens(texto: str) -> int:
    """Documenta: recebe str, retorna int."""
    return len(enc.encode(texto))

---

# 💻 Lab do aluno — versão com lacunas

## 📋 Roteiro do Lab

**Lab — Aula 04 · 2º Semestre · CKP01 R4**  
### Context engineering no chatbot do grupo ★★

*Grupo 3–4 · 20 minutos · Google Colab*

1. Cole o system prompt original do chatbot do grupo (das Aulas 01–03) na Lacuna 1.
2. Reescreva com XML tagging na Lacuna 2: use pelo menos 3 tags (<persona>, <restricoes>, <formato>). Elimine redundâncias.
3. Meça e documente: a célula de medição mostra a redução em tokens automaticamente. Cole o resultado no notebook como comentário.
4. Teste de aderência: use uma pergunta fora do domínio misturada com uma do domínio (ex: "Me explique X. E qual é a capital do Brasil?"). Observe se ambas as versões redirecionam corretamente.

> **🎯 Gabarito das lacunas**
>
> Lacuna 1:  o system prompt atual do grupo, colado exatamente como está desde as Aulas 01–03.
>
> Lacuna 2:  a versão reescrita usando no mínimo as tags de persona, restrições e formato — mais uma tag de contexto, se fizer sentido para o domínio do grupo.
>
> Lacuna 3:  a mensagem humana de cada chain repete o mesmo placeholder de pergunta usado desde a Aula 01, na mesma estrutura de papel e template — idêntica nas duas chains.
>
> Lacuna 4:  a pergunta de teste fica guardada em uma única variável e é usada como valor da mesma chave de entrada nas duas chamadas de invoke, garantindo que antes e depois recebam exatamente a mesma pergunta.

> **💡 Dica de compactação:**
>
> instruções na forma imperativa usam menos tokens. "Seja sempre amigável com o usuário" → "Tom amigável". "Não fale sobre assuntos não relacionados" → "Somente [domínio]".

In [ ]:
!pip install langchain langchain-ollama tiktoken -q

import tiktoken
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import os
from google.colab import userdata
os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")
llm = ChatOllama(model="gpt-oss:120b")

def contar_tokens(texto: str, modelo: str = "gpt-4") -> int:
    return len(tiktoken.encoding_for_model(modelo).encode(texto))

# 👉 LACUNA 1: cole aqui o system prompt original do grupo (Aulas 01–03)
SYSTEM_ORIGINAL = ___

# 👉 LACUNA 2: reescreva com XML tagging — persona, restricoes, formato
SYSTEM_OTIMIZADO = ___  # use <persona>, <restricoes>, <formato>

# Medição automática
tok_antes  = contar_tokens(SYSTEM_ORIGINAL)
tok_depois = contar_tokens(SYSTEM_OTIMIZADO)
print(f"Antes:  {tok_antes} tokens")
print(f"Depois: {tok_depois} tokens")
print(f"Redução: {(1-tok_depois/tok_antes)*100:.1f}%")

# 👉 LACUNA 3: crie 2 chains (antes e depois) e teste com a mesma pergunta
chain_antes  = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_ORIGINAL),
    ("human",  ___),
]) | llm | StrOutputParser()

chain_depois = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_OTIMIZADO),
    ("human",  ___),
]) | llm | StrOutputParser()

# 👉 LACUNA 4: invoke com a mesma pergunta de teste para comparar saídas
pergunta_teste = ___
print("=== ANTES ===\n", chain_antes.invoke({___: pergunta_teste}))
print("=== DEPOIS ===\n", chain_depois.invoke({___: pergunta_teste}))

## 📚 Referências da aula

- Blog Anthropic Engineering — "Effective Context Engineering for AI Agents" (setembro, 2025). A fonte primária do termo e das técnicas desta aula. anthropic.com/engineering/building-effective-agents
- Paper Liu, N. et al. — "Lost in the Middle: How Language Models Use Long Contexts." EMNLP, 2023. Base empírica do context rot. arxiv.org/abs/2307.03172
- Docs Anthropic — Prompt Library e guia de XML tagging. docs.anthropic.com/pt/docs/build-with-claude/prompt-engineering/use-xml-tags
- Docs tiktoken — Biblioteca de tokenização da OpenAI. Funciona como aproximação para qualquer modelo baseado em BPE. github.com/openai/tiktoken
- Livro Russell, S.; Norvig, P. — Inteligência Artificial. 3ª ed. Pearson, 2016. Cap. 22 — A importância do contexto na inferência linguística: a base teórica de por que o contexto é informação.

---

**→ Próxima Aula — Aula 05 · 31/08** — Embeddings e busca semântica com ChromaDB
  
Transformar texto em vetores e buscar por similaridade. A fundação do RAG começa aqui.

---

*Copyright © 2026 Prof. Jorge Luiz Gomes · FIAP · Todos os direitos reservados.*